In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
ANIOS = [2022, 2023, 2024]

df = pd.concat(
    [pd.read_csv(f"victimasFGJ_{a}.csv", low_memory=False).assign(archivo=a) for a in ANIOS],
    ignore_index=True
)
print(df.shape)
df["alcaldia_catalogo"].isna().sum()

validas = df.dropna(subset=["alcaldia_hecho", "alcaldia_catalogo"])
validas = validas[~validas["alcaldia_catalogo"].isin(["CDMX (indeterminada)", "Fuera de CDMX"])]
mapa = validas.groupby("alcaldia_hecho")["alcaldia_catalogo"].agg(lambda s: s.mode().iloc[0])

faltan = df["alcaldia_catalogo"].isna()
df.loc[faltan, "alcaldia_catalogo"] = df.loc[faltan, "alcaldia_hecho"].map(mapa)
df["alcaldia_catalogo"].value_counts(dropna=False)


validas = df.dropna(subset=["alcaldia_hecho", "alcaldia_catalogo"])
validas = validas[~validas["alcaldia_catalogo"].isin(["CDMX (indeterminada)", "Fuera de CDMX"])]
mapa = validas.groupby("alcaldia_hecho")["alcaldia_catalogo"].agg(lambda s: s.mode().iloc[0])

faltan = df["alcaldia_catalogo"].isna()
df.loc[faltan, "alcaldia_catalogo"] = df.loc[faltan, "alcaldia_hecho"].map(mapa)
df["alcaldia_catalogo"].value_counts(dropna=False)

df["calidad_juridica"] = df["calidad_juridica"].replace({"VICTIMA Y DENUNCIANTE": "DENUNCIANTE Y VICTIMA"})
df["competencia"] = df["competencia"].replace({
    "HECHOS NO DELICTIVOS": "HECHO NO DELICTIVO",
    "INCOMPETENCIAS": "INCOMPETENCIA"
})

df.loc[df["edad"] > 100, "edad"] = np.nan   # 17 casos con edades imposibles (hasta 361)
df["fecha_hecho"] = pd.to_datetime(df["fecha_hecho"], errors="coerce")
df["fecha_inicio"] = pd.to_datetime(df["fecha_inicio"], errors="coerce")
df["hora"] = pd.to_datetime(df["hora_hecho"], format="%H:%M:%S", errors="coerce").dt.hour

cdmx = df[df["alcaldia_catalogo"].notna() &
          ~df["alcaldia_catalogo"].isin(["CDMX (indeterminada)", "Fuera de CDMX"])]
print(f"Registros conservados: {len(cdmx):,} de {len(df):,}")
cdmx.to_parquet("victimas_cdmx_limpio.parquet", index=False)





(653915, 23)
Registros conservados: 629,530 de 653,915
